In [18]:
import pandas as pd
import numpy as np

# Import Data 

In [ ]:

df_raw = pd.read_csv("hf://datasets/mindweave/web-server-logs/data/access_logs.csv")
print(f"Dataset chargé : {df_raw.shape[0]} lignes, {df_raw.shape[1]} colonnes")

Dataset chargé : 5000 lignes, 10 colonnes


In [ ]:
print("*"*50)
print ("Data Exploration:")
print("*"*50)
print("1. Data Header:")
print(df_raw.head())
print("2. Data Info:")
df_raw.info()
print("3. Missing Values:")
print(df_raw.isnull().sum())
print("4. Statistical Description:")
print(df_raw.describe())
print("5. Duplicate Rows:")
print(df_raw.duplicated().sum())
print("6. Unique Values per Column:")
for col in df_raw.columns:
    print(f"{col}: {df_raw[col].nunique()} unique values")


**************************************************
Data Exploration:
**************************************************
1. Data Header:
             timestamp  server_id method                      path  \
0  2024-01-01 00:15:12          1    GET  /api/v2/payments/process   
1  2024-01-01 00:27:54          2    GET                     /docs   
2  2024-01-01 00:31:42          1   POST              /api/v2/cart   
3  2024-01-01 00:54:33          2   POST         /static/js/app.js   
4  2024-01-01 01:36:08          2    GET                  /contact   

   status_code  response_time_ms  bytes_sent  \
0          201               153        2256   
1          304               290       10395   
2          200               201        2452   
3          201                31        2324   
4          200                64         449   

                                          user_agent      ip_address  \
0                                 Go-http-client/2.0    107.8.181.20   
1     Bing

In [14]:
print("DISTRIBUTION STATUS CODES")
print(df_raw["status_code"].value_counts().sort_index())

print(" DISTRIBUTION SERVER ID ")
print(df_raw["server_id"].value_counts().sort_index())

print("DISTRIBUTION MÉTHODES HTTP")
print(df_raw["method"].value_counts())

DISTRIBUTION STATUS CODES
status_code
200    3638
201     245
301     118
304     292
400     192
401     156
403      52
404     197
500      85
503      25
Name: count, dtype: int64
 DISTRIBUTION SERVER ID 
server_id
1    1670
2    1642
3    1688
Name: count, dtype: int64
DISTRIBUTION MÉTHODES HTTP
method
GET       3360
POST      1013
PUT        378
DELETE     158
PATCH       91
Name: count, dtype: int64


# **2. NETTOYAGE**

In [ ]:
df = df_raw.copy()

# Convertir timestamp en datetime
df["timestamp"] = pd.to_datetime(df["timestamp"])
print(f"Timestamp converti : {df['timestamp'].dtype}")

# Remplacer les referrer "-" par NaN
df["referrer"] = df["referrer"].replace("-", np.nan)
print(f"Referrer nettoyé : {df['referrer'].isnull().sum()} valeurs manquantes")
# S'assurer que les colonnes numériques sont bien typées
df["response_time_ms"] = pd.to_numeric(df["response_time_ms"], errors="coerce")
df["bytes_sent"]       = pd.to_numeric(df["bytes_sent"], errors="coerce")
df["status_code"]      = pd.to_numeric(df["status_code"], errors="coerce")
print(f"Types corrigés")
print(df.dtypes)

Timestamp converti : datetime64[ns]
Referrer nettoyé : 997 valeurs manquantes
Types corrigés
timestamp           datetime64[ns]
server_id                    int64
method                      object
path                        object
status_code                  int64
response_time_ms             int64
bytes_sent                   int64
user_agent                  object
ip_address                  object
referrer                    object
dtype: object


In [21]:
avant = len(df)
df = df.drop_duplicates()
apres = len(df)
print(f"Doublons supprimés : {avant - apres}")
print(f"Lignes restantes   : {apres}")

Doublons supprimés : 0
Lignes restantes   : 5000


In [23]:
df["referrer"] = df["referrer"].fillna("direct")
print("Valeurs manquantes traitées")
print(df.isnull().sum())

Valeurs manquantes traitées
timestamp           0
server_id           0
method              0
path                0
status_code         0
response_time_ms    0
bytes_sent          0
user_agent          0
ip_address          0
referrer            0
dtype: int64


### 2.2 Suppression des lignes corrompues

In [25]:
# Supprimer les lignes avec response_time négatif ou nul
df = df[df["response_time_ms"] > 0]

# Supprimer les lignes avec bytes_sent négatif
df = df[df["bytes_sent"] >= 0]

# Supprimer les status codes invalides
df = df[df["status_code"].between(100, 599)]

print(f"Lignes corrompues supprimées")
print(f"Lignes finales : {len(df)}")

Lignes corrompues supprimées
Lignes finales : 5000


# **3. PARSING & ENRICHISSEMENT**

### 3.1 Parsing du status code

In [26]:
# Catégorisation des status codes
df["is_success"]   = df["status_code"].between(200, 299).astype(int)
df["is_redirect"]  = df["status_code"].between(300, 399).astype(int)
df["is_error_4xx"] = df["status_code"].between(400, 499).astype(int)
df["is_error_5xx"] = df["status_code"].between(500, 599).astype(int)

# Colonne catégorielle lisible
def categorize_status(code):
    if 200 <= code <= 299: return "success"
    if 300 <= code <= 399: return "redirect"
    if 400 <= code <= 499: return "client_error"
    if 500 <= code <= 599: return "server_error"
    return "unknown"

df["status_category"] = df["status_code"].apply(categorize_status)

print("Status codes parsés")
print(df["status_category"].value_counts())

Status codes parsés
status_category
success         3883
client_error     597
redirect         410
server_error     110
Name: count, dtype: int64


### 3.2 Parsing du user_agent

In [27]:
def classify_agent(ua):
    if pd.isna(ua):
        return "unknown"
    ua = ua.lower()
    bots = ["bot", "crawler", "spider", "curl", "python", "postman", "go-http"]
    if any(b in ua for b in bots):
        return "bot"
    if any(m in ua for m in ["android", "iphone", "mobile", "pixel"]):
        return "mobile"
    if "macintosh" in ua or "windows" in ua:
        return "desktop"
    return "other"

df["client_type"] = df["user_agent"].apply(classify_agent)

print("User agents classifiés")
print(df["client_type"].value_counts())

User agents classifiés
client_type
bot        2975
mobile     1027
desktop     998
Name: count, dtype: int64


### 3.3 Parsing du path (endpoint)

In [ ]:
def classify_path(path):
    if pd.isna(path):
        return "unknown"
    if "/api/" in path:
        return "api"
    if "/static/" in path:
        return "static"
    if any(x in path for x in ["/login", "/register", "/auth"]):
        return "auth"
    if "/admin" in path:
        return "admin"
    return "page"

df["endpoint_type"] = df["path"].apply(classify_path)

# Extraire le nom de l'endpoint API principal
def extract_api_name(path):
    parts = path.strip("/").split("/")
    # /api/v2/payments/process → "payments"
    if len(parts) >= 3 and parts[0] == "api":
        return parts[2] if len(parts) > 2 else parts[-1]
    return "non-api"

df["api_name"] = df["path"].apply(extract_api_name)

print("Endpoints parsés")


Endpoints parsés
endpoint_type
api       2226
page      2012
static     460
auth       302
Name: count, dtype: int64
endpoint_type
api       2226
page      2012
static     460
auth       302
Name: count, dtype: int64


### 3.4 Extraction des features temporelles

In [ ]:
df["hour"] = df["timestamp"].dt.hour
df["minute"] = df["timestamp"].dt.minute
df["dayofweek"]= df["timestamp"].dt.dayofweek      # 0=lundi, 6=dimanche
df["day"] = df["timestamp"].dt.day
df["month"] = df["timestamp"].dt.month
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
df["is_business_hours"] = df["hour"].between(8, 18).astype(int)


print("Features temporelles extraites")
df[["timestamp", "hour", "dayofweek", "is_weekend", "is_business_hours"]].head(5)

Features temporelles extraites


,timestamp,hour,dayofweek,is_weekend,is_business_hours
0,2024-01-01 00:15:12,0,0,0,0
1,2024-01-01 00:27:54,0,0,0,0
2,2024-01-01 00:31:42,0,0,0,0
3,2024-01-01 00:54:33,0,0,0,0
4,2024-01-01 01:36:08,1,0,0,0


In [36]:
print(f"Colonnes après enrichissement : {len(df.columns)}")
print(f"\nListe des colonnes :")
for col in df.columns:
    print(f"  - {col} ({df[col].dtype})")

Colonnes après enrichissement : 25

Liste des colonnes :
  - timestamp (datetime64[ns])
  - server_id (int64)
  - method (object)
  - path (object)
  - status_code (int64)
  - response_time_ms (int64)
  - bytes_sent (int64)
  - user_agent (object)
  - ip_address (object)
  - referrer (object)
  - is_success (int64)
  - is_redirect (int64)
  - is_error_4xx (int64)
  - is_error_5xx (int64)
  - status_category (object)
  - client_type (object)
  - endpoint_type (object)
  - api_name (object)
  - hour (int32)
  - minute (int32)
  - dayofweek (int32)
  - day (int32)
  - month (int32)
  - is_weekend (int64)
  - is_business_hours (int64)


# **4. PARSING & ENRICHISSEMENT**